# Data augmentation for nighttime detection PAPI dataset

## Imports

In [1]:
import os
import cv2
import albumentations as A

## Data augmentation

In [2]:
# =========================================================
# INPUT / OUTPUT DATASET PATHS
# =========================================================

INPUT_DATASET = "PAPI_Night.yolo26"
OUTPUT_DATASET = "PAPI_Night_Augmented.yolo26"

SPLITS = ["train", "valid", "test"]

# =========================================================
# AUGMENTATION PIPELINE
# =========================================================

transform = A.Compose(
    [
        A.RandomBrightnessContrast(p=0.5),
        A.GaussNoise(p=0.3),
        A.MotionBlur(blur_limit=3, p=0.2),
        A.Rotate(limit=5, border_mode=cv2.BORDER_CONSTANT, p=0.5),
        A.HorizontalFlip(p=0.5),
    ],
    bbox_params=A.BboxParams(
        format='yolo',
        label_fields=['class_labels']
    )
)

# =========================================================
# CREATE OUTPUT STRUCTURE
# =========================================================

for split in SPLITS:
    os.makedirs(
        os.path.join(OUTPUT_DATASET, split, "images"),
        exist_ok=True
    )

    os.makedirs(
        os.path.join(OUTPUT_DATASET, split, "labels"),
        exist_ok=True
    )

# =========================================================
# PROCESS EACH SPLIT
# =========================================================

for split in SPLITS:

    input_image_dir = os.path.join(
        INPUT_DATASET,
        split,
        "images"
    )

    input_label_dir = os.path.join(
        INPUT_DATASET,
        split,
        "labels"
    )

    output_image_dir = os.path.join(
        OUTPUT_DATASET,
        split,
        "images"
    )

    output_label_dir = os.path.join(
        OUTPUT_DATASET,
        split,
        "labels"
    )

    print(f"\nProcessing split: {split}")

    for filename in os.listdir(input_image_dir):

        if not filename.lower().endswith((".jpg", ".jpeg", ".png")):
            continue

        image_path = os.path.join(input_image_dir, filename)

        label_path = os.path.join(
            input_label_dir,
            filename.rsplit(".", 1)[0] + ".txt"
        )

        # -------------------------------------------------
        # READ IMAGE
        # -------------------------------------------------

        image = cv2.imread(image_path)

        if image is None:
            print(f"Could not read image: {image_path}")
            continue

        # -------------------------------------------------
        # READ YOLO LABELS
        # -------------------------------------------------

        bboxes = []
        class_labels = []

        if os.path.exists(label_path):

            with open(label_path, "r") as f:

                for line in f.readlines():

                    parts = line.strip().split()

                    if len(parts) != 5:
                        continue

                    class_id = int(parts[0])
                    bbox = list(map(float, parts[1:]))

                    class_labels.append(class_id)
                    bboxes.append(bbox)

        # -------------------------------------------------
        # COPY ORIGINAL IMAGE
        # -------------------------------------------------

        output_original_image = os.path.join(
            output_image_dir,
            filename
        )

        output_original_label = os.path.join(
            output_label_dir,
            filename.rsplit(".", 1)[0] + ".txt"
        )

        cv2.imwrite(output_original_image, image)

        if os.path.exists(label_path):

            with open(label_path, "r") as src:
                original_labels = src.read()

            with open(output_original_label, "w") as dst:
                dst.write(original_labels)

        # -------------------------------------------------
        # CREATE AUGMENTATIONS
        # -------------------------------------------------

        # Creates 2 augmentations -> dataset becomes 3x
        for i in range(2):

            transformed = transform(
                image=image,
                bboxes=bboxes,
                class_labels=class_labels
            )

            aug_image = transformed["image"]
            aug_bboxes = transformed["bboxes"]
            aug_labels = transformed["class_labels"]

            new_filename = (
                f"{filename.rsplit('.',1)[0]}_aug{i}.jpg"
            )

            output_aug_image = os.path.join(
                output_image_dir,
                new_filename
            )

            output_aug_label = os.path.join(
                output_label_dir,
                new_filename.rsplit(".", 1)[0] + ".txt"
            )

            # Save augmented image
            cv2.imwrite(output_aug_image, aug_image)

            # Save augmented labels
            with open(output_aug_label, "w") as f:

                for cls, box in zip(aug_labels, aug_bboxes):

                    x, y, bw, bh = box

                    f.write(
                        f"{cls} {x} {y} {bw} {bh}\n"
                    )

    print(f"Finished split: {split}")

print("\nDataset augmentation complete.")


Processing split: train
Finished split: train

Processing split: valid
Finished split: valid

Processing split: test
Finished split: test

Dataset augmentation complete.
